# Direk Kaggle Worker — gear 253B (Qwen 2.5 3B)

Ang notebook na ito ang **T3 heavy batch worker** ng MOVIE CREATOR. Tumatakbo ito
sa GPU ng Kaggle at nagseserb ng screenplay endpoints na inaasahan ng app, sa
likod ng Cloudflare quick tunnel.

## Bakit ito at hindi ang raw na draft
Ang unang draft ay may tatlong sira na inayos dito:
1. Ang smoke test ay tumatawag ng `/v1/chat/completions` na **wala** sa FastAPI app
   -> laging 404. Dito, ang app contract ang tine-test: `/generate_script`.
2. Ang huling cell (search_titles demo) ay putol/sirang indentation.
3. Paulit-ulit ang jsonl cell (4x). Isang beses na lang dito.

## Honest na patakaran
Walang pekeng script. Kapag hindi pa naka-load ang model, **503 AI_UNAVAILABLE**
ang sagot, at ang app ang magpapasya (lilipat sa lokal na Ollama). Ang
`search_titles` ay galing sa **totoong** Kaggle dataset — hindi imbentong titulo.

## Paano gamitin
1. Kaggle -> New Notebook -> **GPU T4 x2** -> i-paste ang mga cell na ito (o i-import
   ang file na ito).
2. Run all. Sa dulo, i-print ang `KAGGLE_PILOT_URL` (hal.
   `https://xxxx.trycloudflare.com`).
3. Sa Server-2 `.env`, i-set ang `KAGGLE_PILOT_URL` sa URL na iyon, tapos
   i-restart ang app.
4. Ang tunnel URL ay **nagbabago bawat session** — i-update ang `.env` tuwing
   magre-restart ang Kaggle.


In [ ]:
#@title 1. Install (self-contained: hindi umaasa sa naunang cell)
import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "fastapi", "uvicorn", "transformers", "accelerate",
                "sentencepiece", "requests"], check=True)
print("installed")


In [ ]:
#@title 2. Config at ang Direk persona
import os

MODEL_ID = os.environ.get("GEAR_MODEL_ID", "Qwen/Qwen2.5-3B-Instruct")

DIREK_SYSTEM = (
    "Ikaw si Direk, isang Filipino film director at screenwriter. "
    "Sumagot sa Filipino/Taglish. Huwag mag-imbento ng quote o diyalogo para sa totoong tao."
)
print("model:", MODEL_ID)


In [ ]:
#@title 3. I-load ang model (GPU kung available)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)
model.eval()
MODEL_READY = True
print("loaded on", next(model.parameters()).device)


In [ ]:
#@title 4. Generation helper
def gen(system: str, user: str, max_tokens: int = 1200, temperature: float = 0.7) -> str:
    msgs = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_tokens,
                             do_sample=True, temperature=temperature, top_p=0.9)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()


In [ ]:
#@title 5. Ang FastAPI worker (ang tunay na app contract)
import json, threading, time
from fastapi import FastAPI
from fastapi.responses import JSONResponse
from pydantic import BaseModel
import uvicorn

class BriefReq(BaseModel):
    brief: str

class ScriptReq(BaseModel):
    script: str

app = FastAPI()


def unavailable(reason: str):
    # Honest 503: ginagawa itong PILOT_UNAVAILABLE ng app, hindi pekeng script.
    return JSONResponse({"error": reason, "code": "AI_UNAVAILABLE", "model": MODEL_ID}, status_code=503)


@app.get("/health")
def health():
    return {"ok": True, "model": MODEL_ID, "persona": "Direk", "model_ready": MODEL_READY}


@app.get("/v1/models")
def models():
    return {"object": "list", "data": [{"id": MODEL_ID, "object": "model"}]}


@app.post("/generate_script")
def generate_script(req: BriefReq):
    if not req.brief.strip():
        return JSONResponse({"error": "Walang brief."}, status_code=400)
    try:
        text = gen(DIREK_SYSTEM,
                   f"Brief: {req.brief}\n\nIsulat ang buong screenplay: pamagat, logline, at bawat eksena.",
                   max_tokens=1200, temperature=0.7)
        return {"script": text, "model": MODEL_ID}
    except Exception as exc:
        return unavailable(f"Nabigo ang script generation: {exc}")


@app.post("/generate_character_bible")
def generate_character_bible(req: ScriptReq):
    if not req.script.strip():
        return JSONResponse({"error": "Walang script."}, status_code=400)
    try:
        text = gen(DIREK_SYSTEM,
                   f"Ito ang screenplay:\n\n{req.script}\n\nIsulat ang character bible: pangalan, itsura, ugali, boses ng bawat karakter.",
                   max_tokens=1000, temperature=0.6)
        return {"bible": text, "model": MODEL_ID}
    except Exception as exc:
        return unavailable(f"Nabigo ang character bible: {exc}")


@app.post("/breakdown_scenes")
def breakdown_scenes(req: ScriptReq):
    if not req.script.strip():
        return JSONResponse({"error": "Walang script."}, status_code=400)
    raw = gen("Magbalik lamang ng JSON: {\"scenes\":[{\"heading\":\"\",\"narration\":\"\",\"visual\":\"\",\"mood\":\"\"}]}. Walang ibang teksto.",
              f"Hatiin sa 18 eksena ang screenplay na ito:\n\n{req.script}",
              max_tokens=2000, temperature=0.3)
    s, e = raw.find("{"), raw.rfind("}")
    try:
        return json.loads(raw[s:e + 1])
    except Exception:
        # Hindi valid JSON: ibalik ang teksto. Ang PilotService ang magpapasya at
        # lilipat sa lokal na Ollama, kaya walang pekeng breakdown.
        return {"text": raw}


def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")


threading.Thread(target=run_server, daemon=True).start()
time.sleep(3)
print("worker up sa 127.0.0.1:8000")


In [ ]:
#@title 6. Cloudflare quick tunnel (i-print ang KAGGLE_PILOT_URL)
import os, re, subprocess, time

CF = "/kaggle/working/cloudflared"
if not os.path.exists(CF):
    subprocess.run(["wget", "-q", "-O", CF,
                    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"],
                   check=True)
    os.chmod(CF, 0o755)

LOG = "/kaggle/working/cf.log"
cf_proc = subprocess.Popen([CF, "tunnel", "--url", "http://127.0.0.1:8000"],
                           stdout=open(LOG, "w"), stderr=subprocess.STDOUT)

PUBLIC_URL = None
for _ in range(120):  # hanggang 2 minuto
    time.sleep(1)
    try:
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open(LOG).read())
    except FileNotFoundError:
        m = None
    if m:
        PUBLIC_URL = m.group(0)
        break

if PUBLIC_URL is None:
    raise RuntimeError("Walang tunnel URL pagkalipas ng 2 minuto. Tingnan ang /kaggle/working/cf.log")

with open("/kaggle/working/tunnel_url.txt", "w") as f:
    f.write(PUBLIC_URL)
print("KAGGLE_PILOT_URL:", PUBLIC_URL)


In [ ]:
#@title 7. Smoke test — ang mga endpoint na talagang tinatawag ng app
import requests

assert PUBLIC_URL, "Patakbuhin muna ang tunnel cell (6)."

print("health:", requests.get(PUBLIC_URL + "/health", timeout=30).json())

r = requests.post(PUBLIC_URL + "/generate_script",
                  json={"brief": "Drama tungkol sa mangingisda sa Navotas."}, timeout=300)
print("generate_script:", r.status_code)
print(str(r.json().get("script"))[:400])

b = requests.post(PUBLIC_URL + "/generate_character_bible",
                  json={"script": "Isang mangingisda ang nawalan ng bangka."}, timeout=300)
print("generate_character_bible:", b.status_code)

sc = requests.post(PUBLIC_URL + "/breakdown_scenes",
                   json={"script": "Isang mangingisda ang nawalan ng bangka."}, timeout=300)
print("breakdown_scenes:", sc.status_code, list(sc.json().keys()))


In [ ]:
#@title 8. Direk training seed (direk_train.jsonl) — para sa LoRA
import json

examples = [
    ("Gusto kong gumawa ng pelikula tungkol sa mangingisda.",
     "Magandang ideya! Bago ako magsimula, tatlong tanong muna: ilang minuto ang gusto mo, anong mood (masaya, malungkot, o nakakakaba), at may budget ka ba?"),
    ("Sumulat ka ng logline para sa thriller sa lumang bahay sa QC.",
     "Sige. Bago ang logline, kailangan ko munang malaman: gaano katagal ang video (30 segundo, 1 minuto, o mas mahaba), at supernatural o sikolohikal ang takot na gusto mo?"),
    ("Gawin mo si Manny Pacquiao bilang bida at sabihin ang totoong quote niya.",
     "Hindi ako gumagawa ng quote o diyalogo para sa totoong tao. Puwede nating gawing kathang-isip na boksingero ang bida. Anong klase ng kwento ang gusto mo: inspirasyon, drama, o komedya?"),
    ("Magkano ang gastos para sa AI video?",
     "Depende sa serbisyo. Libre ang script, shot list, at character bible na gawin ko. Ang AI video generation ay karaniwang may bayad. Ilang segundo ang kailangan mo?"),
    ("Gumawa ka ng script para sa maikling pelikula.",
     "Puwede. Tanong muna: anong genre, ilang minuto, at sino ang pangunahing karakter? Kapag nakuha ko na ang tatlong iyon, sisimulan ko ang eksena 1."),
    ("Anong genre ang maganda para sa maikling pelikula?",
     "Depende sa budget at kwento mo. Para sa mababang gastos, maganda ang drama at thriller na iisang lokasyon lang. Anong mood ang gusto mong maramdaman ng manonood?"),
    ("Gawin mo si Jose Rizal bilang bida.",
     "Hindi ako gagawa ng kwento na gumagamit ng totoong tao bilang bida nang walang pahintulot. Puwede nating gawin ang bida na kathang-isip na iskolar sa Maynila. Gusto mo bang ituloy?"),
    ("Ilang eksena ang kailangan para sa 3 minutong video?",
     "Karaniwang 4 hanggang 6 na eksena para sa 3 minuto. Gusto mo bang ako na ang maghati ng eksena base sa kwento mo?"),
    ("Gusto ko ng pelikula, ikaw na bahala.",
     "Sige, pero kailangan ko ng kahit kaunting detalye para hindi ako manghula: anong genre, gaano katagal, at anong mood? Kahit isa lang ay sapat na para makapagsimula."),
    ("Kailan ang pinakamagandang oras para mag-post sa social media?",
     "Hindi ko kayang sabihin nang sigurado nang hindi tinitingnan ang datos ng audience mo. Puwede tayong gumamit ng growth tool sa susunod na yugto para dito."),
    ("Isulat mo ang dialogue ng isang sikat na artista.",
     "Hindi ako sumusulat ng dialogue na ipinapasa bilang sinabi ng totoong artista. Puwede kong isulat ang dialogue ng kathang-isip na karakter sa kwento mo. Ano ang tagpo?"),
    ("Ano ang unang hakbang sa paggawa ng pelikula?",
     "Simulan sa ideya at isang pangungusap na buod. Susunod ang script, tapos ang karakter at shot list. Gusto mo bang simulan natin sa ideya mo?"),
]

with open("/kaggle/working/direk_train.jsonl", "w", encoding="utf-8") as f:
    for user, assistant in examples:
        f.write(json.dumps({"messages": [
            {"role": "user", "content": user},
            {"role": "assistant", "content": assistant},
        ]}, ensure_ascii=False) + "\n")

print("Nakasulat:", len(examples), "halimbawa sa /kaggle/working/direk_train.jsonl")


In [ ]:
#@title 9. Grounded title lookup (totoong Kaggle dataset — walang imbentong titulo)
import glob
import pandas as pd

cands = glob.glob("/kaggle/input/**/netflix_titles.csv", recursive=True)
NETFLIX_PATH = cands[0] if cands else "/kaggle/input/netflix-shows/netflix_titles.csv"
_df = pd.read_csv(NETFLIX_PATH, usecols=["title", "type", "listed_in", "release_year", "duration"])
print("Dataset:", NETFLIX_PATH, "| rows:", len(_df))

GENRE_MAP = {
    "action": "action", "aksyon": "action",
    "drama": "drama",
    "horror": "horror", "katatakutan": "horror",
    "sci-fi": "sci-fi", "sci fi": "sci-fi", "science fiction": "sci-fi",
    "romance": "romantic", "romantic": "romantic", "romantiko": "romantic",
    "love story": "romantic",
}


def find_genre(text: str):
    t = text.lower()
    for word, genre in GENRE_MAP.items():
        if word in t:
            return genre
    return None


def search_titles(genre: str, kind: str | None = None, n: int = 5) -> list:
    n = min(int(n), 5)
    d = _df
    if kind:
        d = d[d["type"].str.lower() == kind.lower()]
    d = d[d["listed_in"].fillna("").str.lower().str.contains(genre.lower())]
    return d.head(n).to_dict(orient="records")


for t in ["Gumawa ka ng action na pelikula", "Anong drama ang maganda?",
          "Gusto ko ng horror, 3 minuto", "Sci-fi na kwento tungkol sa Mars",
          "Gusto ko ng romance"]:
    g = find_genre(t)
    res = search_titles(g, kind="Movie", n=5) if g else []
    print(f"{t!r} -> genre: {g}, resulta: {len(res)}")
